In [1]:
import numpy as np
from collections import Counter

# -----------------------------------
# Input Corpus
# -----------------------------------
# corpus.txt file
words = ["low", "low", "lowest", "newer", "newest", "wide", "wider"]

words = [word.lower() for word in words]

print("Input Corpus:")
print(words)

# -----------------------------------
# Create Initial Vocabulary
# -----------------------------------
token_dict = {}

for word in words:
    token = " ".join(word) + " </w>"
    token_dict[token] = token_dict.get(token, 0) + 1

print("\n----- Initial Tokens -----")
for token, count in token_dict.items():
    print(token, "->", count)

# -----------------------------------
# Statistics
# -----------------------------------
freq = np.array(list(token_dict.values()))

print("\n----- Statistics -----")
print("Total Words:", np.sum(freq))
print("Highest Frequency:", np.max(freq))


# -----------------------------------
# Count Adjacent Symbol Pairs
# -----------------------------------
def count_pairs(vocabulary):
    pair_count = Counter()

    for token, count in vocabulary.items():
        symbols = token.split()

        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_count[pair] += count

    return pair_count


# -----------------------------------
# Merge Selected Pair
# -----------------------------------
def apply_merge(pair, vocabulary):
    updated_vocab = {}

    for token, count in vocabulary.items():
        symbols = token.split()
        new_symbols = []
        i = 0

        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        updated_token = " ".join(new_symbols)
        updated_vocab[updated_token] = (
            updated_vocab.get(updated_token, 0) + count
        )

    return updated_vocab


# -----------------------------------
# Learn BPE Merge Rules
# -----------------------------------
rules = []
iterations = 10

for step in range(iterations):
    pair_freq = count_pairs(token_dict)

    if not pair_freq:
        break

    best_pair = max(pair_freq, key=pair_freq.get)
    rules.append(best_pair)

    print(f"\nStep {step + 1}")
    print("Merged Pair:", best_pair)
    print("Count:", pair_freq[best_pair])

    token_dict = apply_merge(best_pair, token_dict)


# -----------------------------------
# Final Vocabulary
# -----------------------------------
vocab = set()

for token in token_dict:
    vocab.update(token.split())

print("\n----- Final Vocabulary -----")
print(sorted(vocab))
print("Vocabulary Size:", len(vocab))


# -----------------------------------
# BPE Encoding
# -----------------------------------
def bpe_encode(text, rules):
    tokens = list(text.lower()) + ["</w>"]

    for rule in rules:
        new_tokens = []
        i = 0

        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == rule[0]
                and tokens[i + 1] == rule[1]
            ):
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens


# -----------------------------------
# BPE Decoding
# -----------------------------------
def bpe_decode(tokens):
    return "".join(tokens).replace("</w>", "")


# -----------------------------------
# Testing
# -----------------------------------
samples = ["newest", "lowest", "lower"]

print("\n----- Testing -----")

for sample in samples:
    encoded = bpe_encode(sample, rules)
    decoded = bpe_decode(encoded)

    print("\nWord:", sample)
    print("Encoded:", encoded)
    print("Decoded:", decoded)


# -----------------------------------
# Learned Rules
# -----------------------------------
print("\n----- Learned Rules -----")

for i, rule in enumerate(rules, 1):
    print(f"{i}. {rule}")

print("\nExecution Completed Successfully.")

Input Corpus:
['low', 'low', 'lowest', 'newer', 'newest', 'wide', 'wider']

----- Initial Tokens -----
l o w </w> -> 2
l o w e s t </w> -> 1
n e w e r </w> -> 1
n e w e s t </w> -> 1
w i d e </w> -> 1
w i d e r </w> -> 1

----- Statistics -----
Total Words: 7
Highest Frequency: 2

Step 1
Merged Pair: ('l', 'o')
Count: 3

Step 2
Merged Pair: ('lo', 'w')
Count: 3

Step 3
Merged Pair: ('low', '</w>')
Count: 2

Step 4
Merged Pair: ('e', 's')
Count: 2

Step 5
Merged Pair: ('es', 't')
Count: 2

Step 6
Merged Pair: ('est', '</w>')
Count: 2

Step 7
Merged Pair: ('n', 'e')
Count: 2

Step 8
Merged Pair: ('ne', 'w')
Count: 2

Step 9
Merged Pair: ('e', 'r')
Count: 2

Step 10
Merged Pair: ('er', '</w>')
Count: 2

----- Final Vocabulary -----
['</w>', 'd', 'e', 'er</w>', 'est</w>', 'i', 'low', 'low</w>', 'new', 'w']
Vocabulary Size: 10

----- Testing -----

Word: newest
Encoded: ['new', 'est</w>']
Decoded: newest

Word: lowest
Encoded: ['low', 'est</w>']
Decoded: lowest

Word: lower
Encoded: ['low',